# Phase 5 — Exploratory Data Analysis (EDA)
## Smart Coupon Swap System

This notebook performs exploratory data analysis across all 10 processed datasets in `data/processed/`.

**Objectives:**
- Inspect dataset dimensions and schema integrity.
- Analyze categorical and numerical distributions.
- Study user behavior (browsing dwell time, request and usage rates).
- Evaluate swap proposal settlement and counterparty rating patterns.
- Assess data correlation and outlier presence.

> **Strict Boundary:** This notebook is purely exploratory. No feature engineering, predictive models, or ML algorithms are implemented.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})

DATA_DIR = os.path.join('..', 'data', 'processed')
OUTPUT_DIR = os.path.join('..', 'outputs', 'eda')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Environment initialized. Reading from:', DATA_DIR)

### 1. Load Processed Datasets

In [ ]:
files = {
    'categories': 'categories_clean.csv',
    'brands': 'brands_clean.csv',
    'users': 'users_clean.csv',
    'coupons': 'coupons_clean.csv',
    'user_preferences': 'user_preferences_clean.csv',
    'coupon_views': 'coupon_views_clean.csv',
    'coupon_requests': 'coupon_requests_clean.csv',
    'coupon_usage': 'coupon_usage_clean.csv',
    'swaps': 'swaps_clean.csv',
    'ratings': 'ratings_clean.csv',
}
dfs = {k: pd.read_csv(os.path.join(DATA_DIR, v)) for k, v in files.items()}
for k, v in dfs.items():
    print(f'{k:<18}: {v.shape[0]:>6} rows, {v.shape[1]:>2} columns')

### 2. Category & Brand Distributions

In [ ]:
cats = dfs['categories']
coupons = dfs['coupons']
c_by_cat = coupons.groupby('category_id').size().reset_index(name='count').merge(cats, on='category_id').sort_values('count', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=c_by_cat, x='category_name', y='count', color='#3b82f6')
plt.title('Coupon Distribution Across Categories')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3. User Demographics & Outlier Check

In [ ]:
users = dfs['users']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(users['age'], bins=20, kde=True, ax=axes[0], color='#6366f1')
axes[0].set_title('User Age Distribution')
sns.boxplot(y=users['age'], ax=axes[1], color='#a5b4fc')
axes[1].set_title('User Age Boxplot (Outlier Check)')
plt.tight_layout()
plt.show()
print(users[['age', 'preferred_discount_min', 'preferred_discount_max']].describe())

### 4. Coupon Values & Duration

In [ ]:
coupons['validity_days'] = (pd.to_datetime(coupons['expiry_date']) - pd.to_datetime(coupons['issue_date'])).dt.days
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(coupons['coupon_value'], bins=30, kde=True, ax=axes[0], color='#ec4899')
axes[0].set_title('Coupon Value Distribution')
sns.histplot(coupons['validity_days'], bins=25, kde=True, ax=axes[1], color='#0ea5e9')
axes[1].set_title('Coupon Validity Period (Days)')
plt.tight_layout()
plt.show()

### 5. Swaps & Ratings Analysis

In [ ]:
swaps = dfs['swaps']
ratings = dfs['ratings']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=swaps, x='swap_status', ax=axes[0], palette='viridis', hue='swap_status', legend=False)
axes[0].set_title('Swap Proposal Outcomes')
sns.countplot(data=ratings, x='rating', ax=axes[1], palette='YlOrRd', hue='rating', legend=False)
axes[1].set_title('Rating Score Breakdown (1 to 5 Stars)')
plt.tight_layout()
plt.show()

### 6. Correlation Analysis

In [ ]:
c_views = dfs['coupon_views'].groupby('coupon_id').size().reset_index(name='view_count')
c_reqs = dfs['coupon_requests'].groupby('coupon_id').size().reset_index(name='req_count')
c_usage = dfs['coupon_usage'].groupby('coupon_id').size().reset_index(name='usage_count')

merged = coupons[['coupon_id', 'discount_value', 'minimum_purchase', 'coupon_value']].merge(c_views, on='coupon_id', how='left').fillna(0)
merged = merged.merge(c_reqs, on='coupon_id', how='left').fillna(0)
merged = merged.merge(c_usage, on='coupon_id', how='left').fillna(0)
corr = merged[['discount_value', 'minimum_purchase', 'coupon_value', 'view_count', 'req_count', 'usage_count']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()